In [ ]:
from IPython.display import Javascript
Javascript("""
function ClickConnect(){
console.log("Working");
document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect,60000)
""")

#My Project

### Inspecting all Categories for Refined Labeling

To improve the `assign_multi_class_label` function, let's first get a complete list of all unique categories present in the `category` column of our dataset. This will help us group them more logically and ensure no critical categories are inadvertently labeled as 'safe'.

In [ ]:
!pip install datasets imbalanced-learn nltk tensorflow seaborn

In [ ]:
!pip install tqdm

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import seaborn as sns
import matplotlib.pyplot as plt

from datasets import load_dataset
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

nltk.download('stopwords')

**Dataset**

In [ ]:
from datasets import load_dataset

# Load the whole dataset
dataset = load_dataset('PKU-Alignment/BeaverTails')


# Load the training dataset
train_dataset = load_dataset('PKU-Alignment/BeaverTails', split='330k_train')
test_dataset = load_dataset('PKU-Alignment/BeaverTails', split='330k_test')

train_df = train_dataset.to_pandas()
test_df = test_dataset.to_pandas()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Create Multi-Class Labels

In [ ]:
unique_categories = set()
for index, row in train_df.iterrows():
    for category_name, is_present in row['category'].items():
        if is_present:
            unique_categories.add(category_name)

print("All unique categories present in the dataset:")
print(sorted(list(unique_categories)))

In [ ]:
def assign_multi_class_label(row):
    cat = row['category']

    # --- Label 0: Severe Harm ---
    if any([
        cat.get('violence,aiding_and_abetting,incitement', False),
        cat.get('terrorism,organized_crime', False),
        cat.get('self_harm', False),
        cat.get('child_abuse', False),
        cat.get('sexually_explicit,adult_content', False)
    ]):
        return 0

    # --- Label 1: Non-violent Harm ---
    if any([
        cat.get('non_violent_unethical_behavior', False),
        cat.get('misinformation_regarding_ethics,laws_and_safety', False),
        cat.get('privacy_violation', False),
        cat.get('financial_crime,property_crime,theft', False),
        cat.get('drug_abuse,weapons,banned_substance', False)
    ]):
        return 1

    # --- Label 2: Social / Contextual Harm ---
    if any([
        cat.get('discrimination,stereotype,injustice', False),
        cat.get('hate_speech,offensive_language', False),
        cat.get('controversial_topics,politics', False),
        cat.get('animal_abuse', False)
    ]):
        return 2

    # --- Label 3: Safe (no harmful categories present) ---
    return 3

In [ ]:
label_names = ['Severe Harm', 'Non-violent Harm', 'Social / Contextual Harm', 'Safe']

In [ ]:
print(label_names[0])

In [ ]:
train_df["label"] = train_df.apply(assign_multi_class_label, axis=1)
test_df["label"] = test_df.apply(assign_multi_class_label, axis=1)

In [ ]:
print("Train distribution:")
print(train_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

In [ ]:
train_df['full_text'] = train_df['prompt'] + " " + train_df['response']
test_df['full_text'] = test_df['prompt'] + " " + test_df['response']

print("First 5 rows of train_df with 'full_text' column:")
display(train_df[['prompt', 'response', 'full_text']].head())

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='label', data=train_df, palette='viridis')
plt.title('Distribution of Multi-Class Labels in Training Data')
plt.xlabel('Label')
plt.ylabel('Count')
plt.legend(labels=label_names)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='label', data=test_df, palette='viridis')
plt.title('Distribution of Multi-Class Labels in Test Data')
plt.xlabel('Label')
plt.ylabel('Count')
plt.legend(labels=label_names)
plt.show()

Preporocessing

In [ ]:
import nltk
nltk.download('wordnet')

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove punctuation

    tokens = text.split()

    # Keep important words (not all stopwords removed)
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words or word in ["not", "no", "how"]
    ]

    return " ".join(tokens)

In [ ]:
# New column, clean_prompt.
train_df["clean_prompt"] = train_df["full_text"].apply(clean_text)
test_df["clean_prompt"] = test_df["full_text"].apply(clean_text)

In [ ]:
train_df['clean_prompt'][0]

In [ ]:
# Train data
# Used clean_prompt and label as X_train and y_train respectively.
X_train = train_df["clean_prompt"]
y_train = train_df["label"]

# Test data
# Used clean_prompt and label as X_train and y_train respectively.
X_test = test_df["clean_prompt"]
y_test = test_df["label"]

Balanced Training Data

In [ ]:
rus = RandomUnderSampler(random_state=42)

X_resampled, y_resampled = rus.fit_resample(
    X_train.to_frame(),
    y_train
)

X_resampled = X_resampled["clean_prompt"]

print("Balanced class distribution:")
print(pd.Series(y_resampled).value_counts())

In [ ]:
X_resampled

In [ ]:
train_df["clean_prompt"]

In [ ]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

In [ ]:
y_train_final.value_counts()

Balanced Test Data

In [ ]:
rus = RandomUnderSampler(random_state=42)

# Keep original test set
X_test_imbal = X_test.copy()
y_test_imbal = y_test.copy()

# Create balanced test set (for analysis only)
X_test_bal, y_test_bal = rus.fit_resample(
    X_test.to_frame(),
    y_test
    )
X_test_bal = X_test_bal["clean_prompt"]

print("Original Test Distribution:\n", y_test_imbal.value_counts())
print("\nBalanced Test Distribution:\n", pd.Series(y_test_bal).value_counts())

Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,3))),
    ("model", LogisticRegression(max_iter=10000, verbose=1))
])

logistic_model.fit(X_train_final, y_train_final)

log_pred = logistic_model.predict(X_test)
log_probs = logistic_model.predict_proba(X_test)

In [ ]:
log_pred = logistic_model.predict(X_test)
log_probs = logistic_model.predict_proba(X_test)

In [ ]:
print("Logistic Regression")
print(classification_report(y_test, log_pred))

In [ ]:
print("Logistic Regression - new")
print(classification_report(y_test, log_pred))

In [ ]:
cm = confusion_matrix(y_test, log_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")

plt.show()

SVC

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

svc_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000)),
    ("model", LinearSVC(class_weight="balanced"))
])

svc_model.fit(X_train_final, y_train_final)

svc_pred = svc_model.predict(X_test)
svc_scores = svc_model.decision_function(X_test)


In [ ]:
cm_svc = confusion_matrix(y_test, svc_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm_svc, annot=True, fmt="d", cmap="Oranges")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Linear SVC")

plt.show()

LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

In [ ]:
max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)

In [ ]:
# Balanced test
X_test_bal_seq = tokenizer.texts_to_sequences(X_test_bal)
X_test_bal_pad = pad_sequences(X_test_bal_seq, maxlen=max_length)

In [ ]:
label_encoder = LabelEncoder()

y_train_lstm = label_encoder.fit_transform(train_df["label"])
y_test_lstm = label_encoder.transform(test_df["label"])

In [ ]:
lstm_model_imb = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_length),

    LSTM(128),
    Dropout(0.5),

    Dense(64, activation='relu'),
    Dense(4, activation='softmax')
])

In [ ]:
lstm_model_imb.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


LSTM Model Training and Validation Loss

In [ ]:
lstm_model_imb.build(input_shape=(None, max_length))
lstm_model_imb.summary()

In [ ]:
from tqdm.keras import TqdmCallback

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = lstm_model_imb.fit(
     X_train_pad,
    y_train_lstm,
    epochs=50,
    batch_size=8192,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
lstm_probs = lstm_model_imb.predict(X_test_pad)
lstm_pred = np.argmax(lstm_probs, axis=1)

In [ ]:
import matplotlib.pyplot as plt

# Get the training and validation loss from the history object
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)

# Plot training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation Loss for LSTM Model')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
cm_lstm = confusion_matrix(y_test_lstm, lstm_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm_lstm, annot=True, fmt="d", cmap="Purples")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - LSTM")

plt.show()

In [ ]:
# Prediction of balanced set
y_pred_lr_bal = logistic_model.predict(X_test_bal)
y_pred_svc_bal = svc_model.predict(X_test_bal)
#y_pred_lstm_bal = np.argmax(lstm_model_imb.predict(X_test_bal_pad), axis=1)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

def plot_cm(y_true, y_pred, title):
  sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
  ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
  plt.title(title)
  plt.show()

# Logistic Regression
plot_cm(y_test, log_pred, "LR - Imbalanced")
plot_cm(y_test_bal, y_pred_lr_bal, "LR - Balanced")

# SVC
plot_cm(y_test, svc_pred, "SVC - Imbalanced")
plot_cm(y_test_bal, y_pred_svc_bal, "SVC - Balanced")

# LSTM
plot_cm(y_test_lstm, lstm_pred, "LSTM - Imbalanced")
plot_cm(y_test_bal, y_pred_lstm_bal, "LSTM - Balanced")

In [ ]:
print("Balanced classification report")
print("Logistic Regression")
print(classification_report(y_test, log_pred))

print("SVC")
print(classification_report(y_test, svc_pred))

# print("LSTM")
# print(classification_report(y_test_lstm, lstm_pred))

In [ ]:

print("Imbalanced classification report")
print("Logistic Regression")
print(classification_report(y_test_bal, y_pred_lr_bal))

print("SVC")
print(classification_report(y_test_bal, y_pred_svc_bal))

# print("LSTM")
# print(classification_report(y_test_bal, y_pred_lstm_bal))

In [ ]:
classes = [0,1,2,3]

y_test_bin = label_binarize(y_test, classes=classes)

log_auc = roc_auc_score(y_test_bin, log_probs, multi_class="ovr")
svc_auc = roc_auc_score(y_test_bin, svc_scores, multi_class="ovr")
lstm_auc = roc_auc_score(y_test_bin, lstm_probs, multi_class="ovr")

print("Logistic AUC:", log_auc)
print("SVC AUC:", svc_auc)
print("LSTM AUC:", lstm_auc)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
plt.figure(figsize=(8,6))

fpr, tpr, _ = roc_curve(y_test_bin.ravel(), log_probs.ravel())
plt.plot(fpr, tpr, label="Logistic")

fpr, tpr, _ = roc_curve(y_test_bin.ravel(), svc_scores.ravel())
plt.plot(fpr, tpr, label="SVC")

fpr, tpr, _ = roc_curve(y_test_bin.ravel(), lstm_probs.ravel())
plt.plot(fpr, tpr, label="LSTM")

plt.plot([0,1],[0,1],"k--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()

plt.show()

In [ ]:
results = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "SVC",
        "LSTM"
    ],

    "ROC-AUC":[
        log_auc,
        svc_auc,
        lstm_auc
    ]
})

results

In [ ]:
sns.barplot(
    x="Model",
    y="ROC-AUC",
    data=results
)

plt.title("Model Performance Comparison")
plt.xticks(rotation=30)

plt.show()